# APEX Optimizer Tutorial: Math Problem Solving

This tutorial demonstrates how to use **APEX** (Analysis-based Prompt Engineering eXpert) to improve GPT-5 Mini's performance on AIME math problems through systematic prompt optimization.

APEX analyzes failures, recognizes success patterns, generates hypotheses, and validates improvements empirically.

## Configuration

All modifiable parameters in one place for easy adjustment:

In [1]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="Pydantic serializer warnings:",
    category=UserWarning,
    module="pydantic.main",
)

In [2]:
from dspy.teleprompt.apex.litellm_session_pool import set_pool_size_for_litellm_session

# API Configuration
api_key = 'sk-12345'# Will prompt if not set
base_url = 'https://nexus-master.lmndstaging.com'

# Student Model Configuration (model being optimized)
student_model = "litellm_proxy/openai/gpt-5-mini"
student_base_url = base_url  # Optional custom API endpoint
student_temperature = 0.0  # Deterministic for math
student_reasoning_effort = 'minimal'

# Analysis Model Configuration (for failure analysis and hypotheses)
# analysis_model = "litellm_proxy/openai/gpt-5"
analysis_model = "litellm_proxy/vertex_ai/gemini-2.5-pro"
analysis_base_url = base_url  # Optional custom API endpoint
analysis_temperature = 1.0  # Creative for hypothesis generation
analysis_reasoning_effort = None#'minimal'

# APEX Optimization Settings
max_iterations = 50
num_hypotheses = 3
train_sample_size = 10
success_threshold = 1.0
convergence_patience = 10
num_threads = 50
seed = 42
verbosity = "detailed"
candidate_selection = 'best_on_val' #"pareto"

# MLflow Tracking (Optional)
use_mlflow = True
mlflow_tracking_uri = "http://localhost:5005"
mlflow_experiment_name = "APEX-AIME-Math"

set_pool_size_for_litellm_session(pool_size=num_threads)

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

Import dependencies and configure language models:

In [3]:
import os
import dspy
from dspy.adapters import JSONAdapter

if api_key is None:
    api_key = os.getenv("OPENAI_API_KEY") or input("Enter your OpenAI API key: ")

# Configure student model
student_kwargs = {
    "model": student_model,
    "api_key": api_key,
    "temperature": student_temperature,
}
if student_base_url:
    student_kwargs["base_url"] = student_base_url
if student_reasoning_effort:
    student_kwargs["reasoning_effort"] = student_reasoning_effort

student_lm = dspy.LM(**student_kwargs)

# Configure analysis model
analysis_kwargs = {
    "model": analysis_model,
    "api_key": api_key,
    "temperature": analysis_temperature,
}
if analysis_base_url:
    analysis_kwargs["base_url"] = analysis_base_url
if analysis_reasoning_effort:
    analysis_kwargs["reasoning_effort"] = analysis_reasoning_effort

analysis_lm = dspy.LM(**analysis_kwargs)

analysis_adapter = JSONAdapter()
hypothesis_adapter = JSONAdapter()

dspy.configure(lm=student_lm)
dspy.settings.configure(num_threads=num_threads)

## Dataset

Load AIME problems (American Invitational Mathematics Examination):

In [4]:
from datasets import load_dataset
import random

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            "solution": x['solution'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

train_set, val_set, test_set = init_dataset()

print(f"Training: {len(train_set)} | Validation: {len(val_set)} | Test: {len(test_set)}")

Training: 45 | Validation: 45 | Test: 150


Example problem:

In [5]:
print("Problem:", train_set[0]['problem'])
print("\nAnswer:", train_set[0]['answer'])

Problem: In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.

Answer: 242


## Program

Define a Chain of Thought program:

In [6]:
from typing import Literal


class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

# program = dspy.ChainOfThought(GenerateResponse)

class RouteResponse(dspy.Signature):
    """Route the problem to the appropriate brain."""
    problem: str = dspy.InputField(description="The problem to solve")
    brain_name: Literal['geometric', 'general'] = dspy.OutputField(description="The name of the brain that should solve the problem.")

class AIMESolver(dspy.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.router = dspy.ChainOfThought(RouteResponse)
        self.geometric_solver = dspy.ChainOfThought(GenerateResponse)
        self.general_solver = dspy.ChainOfThought(GenerateResponse)

    def forward(self, problem: str) -> dspy.Prediction:
        route = self.router(problem=problem)
        if route.brain_name == 'geometric':
            return self.geometric_solver(problem=problem)
        else:
            return self.general_solver(problem=problem)

program = AIMESolver()


## Metrics

Define evaluation metrics:

In [7]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

In [8]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = (
            f"The final answer must be a valid integer. "
            f"You responded with '{prediction.answer}', which couldn't be parsed. "
            f"The correct answer is '{correct_answer}'."
        )
        
        if written_solution:
            feedback_text += f" Here's the full solution:\n{written_solution}"
        
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)
    
    if score == 1:
        feedback_text = f"Correct! The answer is '{correct_answer}'."
    else:
        feedback_text = f"Incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += f" Here's the full solution:\n{written_solution}"

    return dspy.Prediction(score=score, feedback=feedback_text)

## Baseline Evaluation

Evaluate the unoptimized program:

In [9]:
eval_kwargs = dict(
    num_threads=num_threads,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
)

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    **eval_kwargs,
)

print("Evaluating baseline...")
baseline_result = evaluate(program)

print(f"\nBaseline Performance: {baseline_result.score / 100.:.1%}")

Evaluating baseline...
Average Metric: 80.00 / 150 (53.3%): 100%|██████████| 150/150 [00:03<00:00, 45.10it/s]

2025/10/21 14:01:33 INFO dspy.evaluate.evaluate: Average Metric: 80 / 150 (53.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b = b+7 and 97_b = 9b+7. We need b+7 to divide 9b+...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"Set up affine coordinates with A=(0,0), B=(1,0), C=(0,1). Points o...",441,✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We must count assignments of 3 labeled flavors (C, V, S) to 9 dist...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer solutions (x,y) in [-100,100] satisfying 12x^2 - x...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Divisibility by 22 means divisible by 2 and 11. Units digit must b...,279,✔️ [1]



Baseline Performance: 53.3%


## APEX Optimization

Optimize the program with APEX:

In [10]:
from tqdm.contrib.logging import logging_redirect_tqdm
from dspy.teleprompt.apex import APEX

optimizer = APEX(
    metric=metric_with_feedback,
    analysis_lm=analysis_lm,
    hypothesis_lm=analysis_lm,
    analysis_adapter=analysis_adapter,
    hypothesis_adapter=hypothesis_adapter,
    max_iterations=max_iterations,
    num_hypotheses=num_hypotheses,
    num_eval_runs=1,
    train_sample=train_sample_size,
    success_threshold=success_threshold,
    convergence_patience=convergence_patience,
    num_threads=num_threads,
    verbosity=verbosity,
    seed=seed,
    candidate_selection=candidate_selection,
    use_mlflow=use_mlflow,
    mlflow_tracking_uri=mlflow_tracking_uri,
    mlflow_experiment_name=mlflow_experiment_name,
)

print("Starting optimization...")

with logging_redirect_tqdm():
    optimized_program = optimizer.compile(
        student=program,
        trainset=train_set,
        valset=val_set,
    )

print("\nOptimization complete!")

2025/10/21 14:01:33 INFO mlflow.tracking.fluent: Experiment with name 'APEX-AIME-Math' does not exist. Creating a new experiment.
2025/10/21 14:01:33 INFO dspy.teleprompt.apex.apex: APEX: MLflow tracking enabled
2025/10/21 14:01:34 INFO dspy.teleprompt.apex.apex: APEX: running with num_threads=50
2025/10/21 14:01:34 INFO dspy.teleprompt.apex.apex: APEX: Configuration - max_iterations=50, num_hypotheses=3, success_threshold=1.00, convergence_patience=10
2025/10/21 14:01:34 INFO dspy.teleprompt.apex.apex: APEX: Using seed=42 for reproducibility
2025/10/21 14:01:34 INFO dspy.teleprompt.apex.apex: APEX: Evaluating initial baseline on validation set


Starting optimization...
Processed 45 / 45 examples: 100%|██████████| 45/45 [00:02<00:00, 15.42it/s]

2025/10/21 14:01:37 INFO dspy.teleprompt.apex.apex: APEX: Initial baseline score=0.5111



Processed 1 / 10 examples:  10%|█         | 1/10 [00:22<03:19, 22.15s/it]

2025/10/21 14:02:00 ERROR dspy.utils.parallelizer: Error for _AnalysisTask(mode='failure', index=0, record=TrainExampleRecord(example=Example({'problem': 'Let $x,$ $y,$ and $z$ be positive real numbers satisfying the system of equations:\n\\begin{align*} \\sqrt{2x-xy} + \\sqrt{2y-xy} &= 1 \\\\ \\sqrt{2y-yz} + \\sqrt{2z-yz} &= \\sqrt2 \\\\ \\sqrt{2z-zx} + \\sqrt{2x-zx} &= \\sqrt3. \\end{align*} \nThen $\\left[ (1-x)(1-y)(1-z) \\right]^2$ can be written as $\\frac{m}{n},$ where $m$ and $n$ are relatively prime positive integers. Find $m+n.$', 'solution': "First, let define a triangle with side lengths $\\sqrt{2x}$, $\\sqrt{2z}$, and $l$, with altitude from $l$'s equal to $\\sqrt{xz}$. $l = \\sqrt{2x - xz} + \\sqrt{2z - xz}$, the left side of one equation in the problem.\nLet  $\\theta$ be angle opposite the side with length $\\sqrt{2x}$. Then the altitude has length $\\sqrt{2z} \\cdot \\sin(\\theta) = \\sqrt{xz}$ and thus $\\sin(\\theta) = \\sqrt{\\frac{x}{2}}$, so $x=2\\sin^2(\\theta)$ 

...

2025/10/21 15:09:22 WARNING dspy.utils.parallelizer: SIGINT received. Cancelling.
2025/10/21 15:09:22 INFO dspy.teleprompt.apex.apex: APEX: Optimization interrupted by user (Ctrl+C)


Processed 9 / 10 examples:  90%|█████████ | 9/10 [01:21<00:09,  9.08s/it]

2025/10/21 15:09:22 INFO dspy.teleprompt.apex.apex: APEX: ✓ Optimization complete | 12 iterations | Reason: interrupted
2025/10/21 15:09:22 INFO dspy.teleprompt.apex.apex: APEX: Final score: 0.6889 (+0.1778 from baseline 0.5111)
2025/10/21 15:09:22 INFO dspy.teleprompt.apex.apex: APEX: Summary - evaluated 33 candidates from 21 hypotheses
2025/10/21 15:09:22 INFO dspy.teleprompt.apex.apex: APEX: Best score trajectory across iterations: [0.5333333333333333, 0.5333333333333333, 0.5777777777777777, 0.5777777777777777, 0.6888888888888889, 0.6888888888888889, 0.6888888888888889, 0.6888888888888889, 0.6888888888888889, 0.6888888888888889, 0.6888888888888889, 0.6888888888888889]



🏃 View run upset-shad-283 at: http://localhost:5005/#/experiments/1/runs/cd827a59bf8f4e93bbae1764b3bfc51c
🧪 View experiment at: http://localhost:5005/#/experiments/1

╔═══════════════════════════════════════════════════════════════╗
║                    APEX Optimization Summary                     ║
╠═══════════════════════════════════════════════════════════════╣
║ Total Iterations: 12                                              ║
║ Hypotheses Tested: 21                                             ║
║ Candidates Evaluated: 33                                          ║
║ Selection Strategy: best_on_val                               ║
╠═══════════════════════════════════════════════════════════════╣
║ Initial Score: 0.5111                                             ║
║ Final Score: 0.6889                                               ║
║ Improvement: +0.1778                                              ║
╠═══════════════════════════════════════════════════════════════╣
║ Iter │ Cand

[Trace(trace_id=tr-ec7c3a055c887438fef837a32678df84), Trace(trace_id=tr-9e4433f2fcd28d570ed11cebf4265111), Trace(trace_id=tr-1bf183dbf59d9555f3bb505b2b64ae40), Trace(trace_id=tr-6fb213260a406412758076fed2601a75), Trace(trace_id=tr-375e8303fd33efcb7e2d4ec32034fbed), Trace(trace_id=tr-9130fcef17be7d8588fa5df21fd2f24e), Trace(trace_id=tr-cac8b46c419fb6cfad01cee974b26221), Trace(trace_id=tr-86edfad36a02deb5cd0bb3c1b61954f9), Trace(trace_id=tr-17db707ce74f6931b55bca7d97da03c7), Trace(trace_id=tr-b03818ba5c11e2023da598914dec2286)]

Inspect the optimized prompt:

In [20]:
print("=" * 50)
# Run throw all named predictors
for predictor_name, predictor in optimized_program.named_predictors():
    print(f"Predictor: {predictor_name}")
    print("=" * 50)
    print(f"{predictor.signature.instructions}")
    print("=" * 50)

Predictor: router.predict
Instructions: Your task is to route a mathematical problem to the most appropriate specialized solver. Analyze the problem statement to determine its primary domain. Pay close attention to the core question; problems asking 'how many', 'count', or about 'probability' are primarily combinatorial and should go to the `general` solver, even if they mention geometric shapes.

Here are the available brains and their specializations:
- `geometric`: Best for problems involving 2D or 3D geometry, trigonometry, spatial reasoning, vector algebra, and dynamic programming.
- `general`: Best for problems involving number theory, algebraic manipulation, solving equations, optimization, combinatorics (counting), probability, and other general mathematical reasoning.

Based on this analysis, choose the single best brain for the given problem.
Predictor: geometric_solver.predict
Instructions: First, think step-by-step to devise a high-level plan for solving the problem. When p

## Final Evaluation

Evaluate the optimized program:

In [12]:
print("Evaluating optimized program...")
optimized_result = evaluate(optimized_program)

print(f"\n{'='*50}")
print(f"Baseline:  {baseline_result.score/100.:.1%}")
print(f"Optimized: {optimized_result.score/100.:.1%}")
print(f"Improvement: {(optimized_result.score - baseline_result.score)/100.:.1%}")
print(f"{'='*50}")

Evaluating optimized program...
Average Metric: 91.00 / 150 (60.7%): : 151it [02:46,  1.10s/it]                       

2025/10/21 15:16:31 INFO dspy.evaluate.evaluate: Average Metric: 91 / 150 (60.7%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We are asked to find integer bases b > 9 such that the base-b numb...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,We are given triangle ABC with points on AB: A - D - E - B with AD...,588,✔️ [1]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,Plan: - We have 9 distinct players; each assigned one of 3 flavors...,16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer pairs (x,y) with -100 ≤ x,y ≤ 100 satisfying 12x^2...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Plan: - An 8-digit number using digits 1–8 exactly once is divisib...,279,✔️ [1]



Baseline:  53.3%
Optimized: 60.7%
Improvement: 7.3%


Trace(trace_id=tr-0a152334a5dc008c8b0dd2df4aeb2c29)

## Optimization Insights

Examine the optimization process:

In [21]:
if hasattr(optimized_program, 'apex_result'):
    result = optimized_program.apex_result
    
    print("Summary:")
    print(f"  Iterations: {len(result.iterations)}")
    print(f"  Candidates evaluated: {len(result.all_candidates)}")
    print(f"  Stop reason: {result.stopped_after}")
    print(f"  Best score: {result.best_candidate.overall_score:.4f}")
    
    print("\nIteration Progress:")
    for it in result.iterations:
        print(f"  Iteration {it.iteration}: {it.num_failures} failures, {len(it.hypotheses)} hypotheses, {len(it.candidates)} candidates")
    
    if result.best_candidate.hypothesis:
        h = result.best_candidate.hypothesis
        print(f"\nBest Hypothesis:")
        print(f"  Strategy: {h.strategy if hasattr(h, 'strategy') else 'N/A'}")
        print(f"  Impact Score: {h.impact_score if hasattr(h, 'impact_score') else 'N/A'}")

Summary:
  Iterations: 12
  Candidates evaluated: 34
  Stop reason: interrupted
  Best score: 0.6889

Iteration Progress:
  Iteration 1: 1 failures, 3 hypotheses, 4 candidates
  Iteration 2: 0 failures, 0 hypotheses, 1 candidates
  Iteration 3: 4 failures, 3 hypotheses, 4 candidates
  Iteration 4: 2 failures, 3 hypotheses, 4 candidates
  Iteration 5: 2 failures, 3 hypotheses, 4 candidates
  Iteration 6: 1 failures, 3 hypotheses, 4 candidates
  Iteration 7: 2 failures, 0 hypotheses, 1 candidates
  Iteration 8: 0 failures, 0 hypotheses, 1 candidates
  Iteration 9: 1 failures, 3 hypotheses, 4 candidates
  Iteration 10: 2 failures, 3 hypotheses, 4 candidates
  Iteration 11: 0 failures, 0 hypotheses, 1 candidates
  Iteration 12: 0 failures, 0 hypotheses, 1 candidates

Best Hypothesis:
  Strategy: Augment the `geometric_solver` prompt by suggesting specific, expert-level geometric heuristics such as considering transformations, symmetries, or coordinate geometry. This provides the model with

## Conclusion

APEX systematically optimizes prompts through:
1. Analyzing failures to understand root causes
2. Recognizing successful patterns
3. Generating data-driven hypotheses
4. Validating improvements empirically

Try adjusting the configuration parameters to explore different optimization strategies.